# Train the model on an alternative training set. Reserve the 2024 catastrophic event for testing.

# BiLSTM Training with AGL Loss 

In [17]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.optimizers import Adam

# ==============================================================
# 🔧 PARÂMETROS FÍSICOS E DE INOVAÇÃO (Ajustados para Estabilidade)
# ==============================================================
LIMIAR_CHEIA_CM = 400.0      # L
THRESHOLD_CM = 2          # θ
PESO_SUBIDA_W = 86.0         # W
ALPHA = 10.0                 # Penalidade nível acima de L
BETA = 18.0                  # Penalidade taxa de subida
GAMMA = 1000.0                # Penalidade sub-previsão em cheia


# === 1️⃣ Carrega e Divide os dados (ESTRATÉGIA DE ISOLAMENTO) ===
dados = np.load("lstm_blumenau_2017_2022_nivel_vazao2.npz", allow_pickle=True)
X_passado, X_futuro, y = dados["X_passado"], dados["X_futuro"], dados["y"].reshape(-1, 1)

# Localizar o maior pico de todos os tempos
indice_pico = np.argmax(y)
print(f"📍 Pico de {y[indice_pico][0]:.2f}cm detectado no índice {indice_pico}")

# Definir janela de teste ao redor do pico (isolando o evento)
margem_antes, margem_depois = 1500, 1200
inicio_test = max(0, indice_pico - margem_antes)
fim_test = min(len(y), indice_pico + margem_depois)

# Conjunto de TESTE (O Evento de 800cm)
X1_test, X2_test, y_test = X_passado[inicio_test:fim_test], X_futuro[inicio_test:fim_test], y[inicio_test:fim_test]

# Conjunto de TREINO + VALIDAÇÃO (Removendo o trecho do pico)
indices_resto = np.setdiff1d(np.arange(len(y)), np.arange(inicio_test, fim_test))
X1_resto, X2_resto, y_resto = X_passado[indices_resto], X_futuro[indices_resto], y[indices_resto]

# Divisão Treino/Validação (80/20) do que sobrou
n_treino = int(len(y_resto) * 0.8)
X1_train, X2_train, y_train = X1_resto[:n_treino], X2_resto[:n_treino], y_resto[:n_treino]
X1_val, X2_val, y_val = X1_resto[n_treino:], X2_resto[n_treino:], y_resto[n_treino:]

# === 2️⃣ Normalização (Ajustada ao Treino) ===
scaler_X1, scaler_X2, scaler_y = MinMaxScaler(), MinMaxScaler(), MinMaxScaler()

X1_train_scaled = scaler_X1.fit_transform(X1_train.reshape(-1, X1_train.shape[2])).reshape(X1_train.shape)
X2_train_scaled = scaler_X2.fit_transform(X2_train.reshape(-1, X2_train.shape[2])).reshape(X2_train.shape)
y_train_scaled = scaler_y.fit_transform(y_train)

X1_val_scaled = scaler_X1.transform(X1_val.reshape(-1, X1_val.shape[2])).reshape(X1_val.shape)
X2_val_scaled = scaler_X2.transform(X2_val.reshape(-1, X2_val.shape[2])).reshape(X2_val.shape)
y_val_scaled = scaler_y.transform(y_val)

# Aplicando a transformação no conjunto do PICO EXTREMO
X1_test_scaled = scaler_X1.transform(X1_test.reshape(-1, X1_test.shape[2])).reshape(X1_test.shape)
X2_test_scaled = scaler_X2.transform(X2_test.reshape(-1, X2_test.shape[2])).reshape(X2_test.shape)

# === 🔁 Conversores de Escala ===
L_scaled = float(scaler_y.transform([[LIMIAR_CHEIA_CM]])[0][0])
theta_scaled = float(scaler_y.transform([[y_train.min() + THRESHOLD_CM]])[0][0] - scaler_y.transform([[y_train.min()]])[0][0])

# === 3️⃣ Função de Perda Customizada ===
def flood_weighted_loss_momento(alpha=ALPHA, beta=BETA, gamma=GAMMA, L=L_scaled, theta=theta_scaled, W=PESO_SUBIDA_W):
    def loss(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
        y_pred = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
        dy = tf.concat([[0.0], y_true[1:] - y_true[:-1]], axis=0)
        
        wi = 1.0 + W * tf.cast(dy > theta, tf.float32)
        aceleracao_term = beta * tf.nn.relu(dy) * tf.square(y_true - y_pred)
        base_term = 1.0 + alpha * tf.nn.relu(y_true - L)
        mse_term = tf.square(y_true - y_pred)
        under_term = gamma * tf.cast(y_true > L, tf.float32) * tf.square(tf.nn.relu(y_true - y_pred))

        numerador = tf.reduce_sum(wi * (base_term * mse_term + under_term + aceleracao_term))
        denominador = tf.reduce_sum(wi) + 1e-7
        return numerador / denominador
    return loss

# === 4️⃣ Construção e Treino ===
in1, in2 = Input(shape=(X1_train.shape[1], X1_train.shape[2])), Input(shape=(X2_train.shape[1], X2_train.shape[2]))
x1 = LSTM(64)(in1)
x2 = LSTM(32)(in2)
c = Concatenate()([x1, x2])
x = Dense(64, activation="relu")(c)
x = Dense(32, activation="relu")(x)
out = Dense(1)(x) # Sem ativação para permitir extrapolação acima de 1.0

modelo_flood1 = Model(inputs=[in1, in2], outputs=out)
modelo_flood1.compile(optimizer=Adam(learning_rate=0.001), loss=flood_weighted_loss_momento())

modelo_flood1.fit([X1_train_scaled, X2_train_scaled], y_train_scaled,
                 validation_data=([X1_val_scaled, X2_val_scaled], y_val_scaled),
                 epochs=100, batch_size=32, verbose=1)

# === 5️⃣ Avaliação no PICO HISTÓRICO ===
y_pred_scaled = modelo_flood1.predict([X1_test_scaled, X2_test_scaled])
y_pred_raw = scaler_y.inverse_transform(y_pred_scaled)

print(f"\n🔥 RESULTADO NO EVENTO EXTREMO (PICO > 800cm):")
print(f"MAE: {mean_absolute_error(y_test, y_pred_raw):.2f}")
print(f"R²: {r2_score(y_test, y_pred_raw):.2f}")

📍 Pico de 939.50cm detectado no índice 46784
Epoch 1/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.5877 - val_loss: 0.0054
Epoch 2/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.0897 - val_loss: 0.0179
Epoch 3/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0581 - val_loss: 0.0081
Epoch 4/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0495 - val_loss: 0.0117
Epoch 5/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.1070 - val_loss: 0.0060
Epoch 6/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0501 - val_loss: 0.0086
Epoch 7/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.0495 - val_loss: 0.0044
Epoch 8/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 0.0366 - val_loss: 0.0134
Epoch 9/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.0317 - val_loss: 0.0074
Epoch 10/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0303 - val_loss: 0.0168
Epoch 11/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 

# BiLSTM Training with MSE Loss

In [ ]:
# 1. Definição da Arquitetura (Idêntica ao LTO para comparação justa)
in1_base = Input(shape=(X1_train.shape[1], X1_train.shape[2]), name="In_Passado_Base")
x1_b = LSTM(64)(in1_base)
x1_b = Dropout(0.2)(x1_b)

in2_base = Input(shape=(X2_train.shape[1], X2_train.shape[2]), name="In_Futuro_Base")
x2_b = LSTM(32)(in2_base)
x2_b = Dropout(0.2)(x2_b)

c_b = Concatenate()([x1_b, x2_b])
x_b = Dense(64, activation="relu")(c_b)
x_b = Dense(32, activation="relu")(x_b)
out_base = Dense(1, name="Out_Base")(x_b)

modelo_base1 = Model(inputs=[in1_base, in2_base], outputs=out_base)

# 2. Compilação com a perda padrão (Mean Squared Error)
modelo_base1.compile(optimizer=Adam(learning_rate=0.001), loss="mse", metrics=["mae"])

print("🚀 Iniciando treinamento do Modelo Base (MSE)...")

# 3. Treinamento
history_base1 = modelo_base1.fit(
    [X1_train_scaled, X2_train_scaled], y_train_scaled,
    validation_data=([X1_val_scaled, X2_val_scaled], y_val_scaled),
    epochs=100, 
    batch_size=32, 
    verbose=1
)

# 4. Predição no PICO HISTÓRICO
y_pred_scaled_base = modelo_base1.predict([X1_test_scaled, X2_test_scaled])
y_p_base_pico = scaler_y.inverse_transform(y_pred_scaled_base).flatten()

print("\n✅ Modelo Base treinado e predições geradas para o pico de 800cm.")

🚀 Iniciando treinamento do Modelo Base (MSE)...
Epoch 1/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - loss: 0.0028 - mae: 0.0345 - val_loss: 7.0389e-04 - val_mae: 0.0194
Epoch 2/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 8.1729e-04 - mae: 0.0194 - val_loss: 6.2298e-04 - val_mae: 0.0181
Epoch 3/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 7.2934e-04 - mae: 0.0181 - val_loss: 5.9353e-04 - val_mae: 0.0176
Epoch 4/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 6.9813e-04 - mae: 0.0172 - val_loss: 5.6824e-04 - val_mae: 0.0173
Epoch 5/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 6.3415e-04 - mae: 0.0164 - val_loss: 6.2873e-04 - val_mae: 0.0181
Epoch 6/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 6.6459e-04 - mae: 0.0166 - val_loss: 6.1870e-04 - val_mae: 0.0186
Epoch 7/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 5.8568e-04 - mae: 0.0157 - val_loss: 6.6786e-04 - val_mae: 0.0186
Epoch 8/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/ste

# Comparative Plot: MSE Loss vs. AGL Flood Loss (AGL-Loss)

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==============================================================
# 1️⃣ GERAÇÃO DAS PREDIÇÕES (Usando os modelos salvos na memória)
# ==============================================================

# Predição com o Modelo LTO (Custom)
y_p_flood_scaled = modelo_flood1.predict([X1_test_scaled, X2_test_scaled])
y_p_flood_pico = scaler_y.inverse_transform(y_p_flood_scaled).flatten()

# Predição com o Modelo Base (MSE)
y_p_base_scaled = modelo_base1.predict([X1_test_scaled, X2_test_scaled])
y_p_base_pico = scaler_y.inverse_transform(y_p_base_scaled).flatten()

# Dados Reais
y_real_pico = y_test.flatten()

# ==============================================================
# 2️⃣ INTERACTIVE EXTRAPOLATION CHART
# ==============================================================
fig = go.Figure()

# Real Level (The 800cm+ event)
fig.add_trace(go.Scatter(y=y_real_pico, mode='lines', name='Real (Extreme Event)',
                         line=dict(color='black', width=3)))

# Base Model (MSE) - Red
fig.add_trace(go.Scatter(y=y_p_base_pico, mode='lines', name='Base Model (MSE)',
                         line=dict(color='red', width=2, dash='dash')))

# LTO Model (Your Innovation) - Green
fig.add_trace(go.Scatter(y=y_p_flood_pico, mode='lines', name='AGL Model (Custom)',
                         line=dict(color='green', width=2.5)))

# Flood Threshold Line (Reference)
#fig.add_hline(y=LIMIAR_CHEIA_CM, line_dash="dot", line_color="orange", 
#              annotation_text="Flood Threshold", annotation_position="bottom right")


fig.update_layout(
    # 1. Define o tamanho base para todo o texto do gráfico
    font=dict(size=18, color="black"), 
    
    # 2. Ajuste fino: O título
    title=dict(
        text=f"Extrapolation Challenge: AGL vs MSE (Real Peak: {np.max(y_real_pico):.1f} cm) - Itajaí River",
        font=dict(size=24) 
    ),
    
    # 3. Eixo X (corretamente aberto e fechado)
    xaxis=dict(
        range=[1300, 1700],
        title="Time in Event Window (Hours/Samples)"
    ), # <--- Veja a vírgula e o fechamento aqui!
    
    # 4. Outras configurações (fora do xaxis)
    yaxis_title="Level (cm)",
    hovermode='x unified',
    template='plotly_white',
    
    # 5. Legenda
    legend=dict(
        yanchor="top", 
        y=0.99, 
        xanchor="left", 
        x=0.01,
        font=dict(size=18)
    )
)
# ==============================================================
# 2.1 INSERIR MARCAÇÃO DE LAG (Linha Violeta)
# ==============================================================

# Defina os pontos onde a linha deve aparecer
# Ajuste estes valores conforme os pontos específicos que deseja destacar
y_level = 350     # Altura (cm) onde a linha será desenhada
x_inicio = 1464   # Ponto de início do lag (ex: onde o modelo Base cruza)
x_fim = 1476      # Ponto de fim do lag (ex: onde o modelo AGL cruza)

fig.add_shape(
    type="line",
    x0=x_inicio, y0=y_level,
    x1=x_fim, y1=y_level,
    line=dict(
        color="purple", 
        width=5  # Espessura da linha
    )
)
# 1. Calculando o valor do Lag (diferença)
lag_value = abs(x_fim - x_inicio)

# 2. Adicionando o texto (rótulo)
fig.add_annotation(
    x=(x_inicio + x_fim) / 2,   # Posiciona no meio da linha
    y=y_level + 20,             # Posiciona um pouco acima da linha (offset de 20 unidades)
    text=f"Lag: {lag_value}h",  # O texto que aparece
    showarrow=False,            # Remove a setinha, deixando apenas o texto
    font=dict(
        size=16, 
        color="purple",
        family="Arial, sans-serif"
    ),
    bgcolor="white",            # Fundo opcional para destacar o texto
    opacity=0.8
)
fig.show()
fig.write_image("peak_criticalevents_itajai.pdf", width=2000, height=550)

# ==============================================================
# 3️⃣ QUICK PEAK METRICS
# ==============================================================
max_real = np.max(y_real_pico)
max_lto = np.max(y_p_flood_pico)
max_base = np.max(y_p_base_pico)

print(f"\n📊 EXTRAPOLATION SUMMARY:")
print(f"Real Peak: {max_real:.1f} cm")
print(f"Peak reached by LTO: {max_lto:.1f} cm (Error: {abs(max_real-max_lto):.1f} cm)")
print(f"Peak reached by Base: {max_base:.1f} cm (Error: {abs(max_real-max_base):.1f} cm)")